# The Aizawa Attractor engine
Exercises the formulary end to end. The attractor is a rotation at the constant rate d with a radius driven by dr/dt = (z-b) r.

In [1]:
import sys, os, numpy as np
sys.path.insert(0, os.path.abspath('../../..'))
from ValaQuenta.modules.aizawa_attractor import maths as M
from ValaQuenta.modules.aizawa_attractor.tools import AizawaAttractorModule
m = AizawaAttractorModule(); print([e.name for e in m.formulary()])

['field', 'cylindrical', 'fixed_points', 'orbit', 'lyapunov', 'js_layout']


## Cylindrical form is exact
dr/dt = (z-b) r and dphi/dt = d, from the field and from the closed form.

In [2]:
o = M.orbit(3000)
res = [abs(a-b) for s in o[::300] for a, b in zip(*M.cylindrical_rates(s))]
print('max residual over the orbit:', max(res))
print(m.run('cylindrical', {})['result'])

max residual over the orbit: 8.881784197001252e-16
{'from_field': (np.float64(0.17088007490635065), np.float64(3.5000000000000004)), 'closed_form': (0.17088007490635068, 3.5)}


## Axisymmetric reduction at f = 0
The 3-D orbit and the 2-D (r, z) flow agree.

In [3]:
p0 = dict(M.STANDARD, f=0.0); s = np.array([0.1, 0, 0.0])
for _ in range(4000): s = M.rk4_step(s, 0.0045, p0)
red = M.reduced_orbit(1, 3999, 0.0045, p0, (0.1, 0.0))[0]
print('3-D (r, z):', [float(np.hypot(s[0], s[1])), float(s[2])], ' reduced:', red.tolist(), ' diff', float(np.abs(np.array([np.hypot(s[0], s[1]), s[2]]) - red).max()))

3-D (r, z): [0.6152948825757664, -0.266881503615537]  reduced: [0.6152948658000615, -0.26688150328821675]  diff 1.6775704825278126e-08


## Fixed points sit on the axis

In [4]:
for f in M.fixed_points(): print('z* =', round(float(f['point'][2]), 4), ' eigenvalues', [complex(round(v.real, 3), round(v.imag, 3)) for v in f['eigenvalues']])

z* = 1.9433  eigenvalues [(1.243+3.5j), (1.243-3.5j), (-2.826+0j)]
z* = -1.105  eigenvalues [(-1.805+3.5j), (-1.805-3.5j), (-0.271+0j)]
z* = -0.8382  eigenvalues [(-1.538+3.5j), (-1.538-3.5j), (0.247+0j)]


## Chaos: the largest Lyapunov exponent

In [5]:
print('lambda_1 =', M.lyapunov_max(steps=30000))

lambda_1 = 0.08481004742930277


## Open items
- The original paper reference for the attractor is not yet verified against a title page.
- Nothing here is a claim about the zeta function; the comparison lives in `zeta_uns_path`.

In [6]:
print('bounds', [x.round(2).tolist() for x in M.bounds(o)])

bounds [[-1.31, -1.45, -0.18], [1.38, 1.11, 1.82]]
